In [62]:
from typing import List
from collections import Counter

import re

DEFAULT_UNK = '[UNK]'
DEFAULT_PAD = '[PAD]'


class BPETokenizer():
    """
    Byte Pair Encoding (BPE) tokenizer implementation.

    BPE is a subword tokenization algorithm that iteratively merges the most frequent
    pairs of characters or character sequences to build a vocabulary of subwords.
    """

    def __init__(
            self,
            vocab_size=8192,
            # percent_bpe=0.2, # todo should we support it?
            UNK=DEFAULT_UNK,
            PAD=DEFAULT_PAD,
        ):
        """
        Initialize the BPE tokenizer.

        Args:
            vocab_size: Maximum size of the vocabulary
            UNK: Token to use for unknown tokens
            PAD: Token to use for padding

        Raises:
            ValueError: If vocab_size is less than 1
        """

        if vocab_size < 1:
            raise ValueError('vocab size must be greater than 0.')

        self.vocab_size = vocab_size

        self.unk = UNK
        self.pad = PAD

        self.vocab = Counter()
        self.merges = {}
        self._token_to_id = None
        self._id_to_token = None

    def _get_stats(self, subwords: List[List[str]]) -> Counter:
        """
        Count frequency of adjacent token pairs in all subwords.

        Args:
            subwords: List of lists, where each inner list contains tokens

        Returns:
            Counter with pairs of adjacent tokens as keys and their frequencies as values
        """
        pairs = Counter()

        # Проходимся по каждому слову (списку токенов)
        for word in subwords:
            # Если слово состоит хотя бы из двух токенов, начинаем подсчет пар
            if len(word) >= 2:
                # Проходим по всем возможным парам токенов внутри одного слова
                for i in range(len(word) - 1):
                    # Создаем кортеж из текущей пары токенов
                    current_pair = (word[i], word[i+1])
                    # Добавляем эту пару в Counter
                    pairs[current_pair] += 1

        return pairs


    def _perform_merge(self, pairs: Counter, subwords: List[List[str]], new_token: str, best_pair: tuple[str, str]) -> List[List[str]]:
        """
        Merge the most frequent pair in all subwords and update pair statistics.

        Args:
            pairs: Counter with token pair frequencies
            subwords: List of lists of tokens to update
            new_token: The merged token to insert
            best_pair: The pair of tokens to merge

        Returns:
            Tuple of (updated pairs counter, updated subwords list)
        """

        # Note, there are two ways to implement this function:
        # 1. Calculate `new_subwords` and update `pairs` Counter on the fly
        #    This way is harder to implement, but more efficient
        # 2. Calculate `new_subwords` and calculate `pairs` from `new_subwords`
        #    This way much more computationally expensive, but simple


        # 1. Hard way
        new_pairs = pairs
        # Easy way
        new_pairs = Counter()


        # Update subwords by merging the best pair
        new_subwords = []

        # For each word, replace occurrences of the best pair with the new token
        for word in subwords:
            new_word = []
            i = 0
            while i < len(word):
                # Check if the next two tokens form the best pair
                if i + 1 < len(word) and (word[i], word[i+1]) == best_pair:
                    # Replace the pair with the new token
                    new_word.append(new_token)
                    i += 2  # Skip over both tokens since we replaced them
                else:
                    # Add the current token to the new word
                    new_word.append(word[i])
                    i += 1
            new_subwords.append(new_word)

        # Recalculate the pair frequencies based on the updated subwords
        new_pairs = self._get_stats(new_subwords)

        return new_pairs, new_subwords


    def fit(self, text_corpus: List[str]) -> Counter:
        """
        Train the tokenizer on a corpus of text.

        This method implements the BPE algorithm:
        1. Start with a vocabulary of individual characters
        2. Count frequencies of adjacent character pairs
        3. Merge the most frequent pair and add to vocabulary
        4. Repeat until reaching target vocabulary size

        Args:
            text_corpus: List of text strings to train on

        Returns:
            Counter object containing the vocabulary with frequencies
        """
        # Initialize vocabulary with characters, not bytes
        subwords = []
        for text in text_corpus:
            # Split text into characters
            subword = list(text)
            subwords.append(subword)
            self.vocab.update(subword)

        # Add special tokens
        self.vocab[self.unk] = 1
        self.vocab[self.pad] = 1

        # Calculate how many merges we need to reach target vocab size
        num_merges = self.vocab_size - len(self.vocab)

        # Calculate initial statistics
        pairs = self._get_stats(subwords)

        # Perform merges until we reach target vocab size
        for i in range(num_merges):

            # Stop if no pairs left
            if not pairs:
                break

            # Find the most common pair
            best_pair = pairs.most_common(1)[0][0]
            best_freq = pairs.most_common(1)[0][1]

            # Stop if no pairs left
            if best_freq < 2:
                break

            # Create new token by merging the best pair
            new_token = ''.join(best_pair)

            # Update vocabulary
            self.vocab[new_token] = best_freq

            # Store the merge operation
            self.merges[best_pair[0]] = best_pair[1]

            # Update subwords by performing the merge
            pairs, subwords = self._perform_merge(pairs, subwords, new_token, best_pair)
            # Recompute stats after merge

            # Create new token by merging the best pair
            new_token = ''.join(best_pair)

        return self.vocab

    def tokenize(self, text: str) -> List[str]:
        """
        Tokenize text into subwords based on the vocabulary.

        This applies the learned BPE merges to the input text:
        1. Start with individual characters
        2. Iteratively apply merges in the order they were learned
        3. Replace unknown tokens with UNK token

        Args:
            text: Input text string

        Returns:
            List of subword tokens

        Raises:
            ValueError: If tokenizer hasn't been fitted yet
        """
        if not self.merges:
            raise ValueError("Tokenizer must be fitted before tokenization")

        # Initialize with individual characters
        subwords = list(text)
        subwords = [subwords]

        # Применяем слияния в порядке их изучения
        for merge in self.merges.items():
            new_subwords = []
            for word in subwords:
                new_word = []
                i = 0
                while i < len(word):
                    # Проверим, является ли текущая пара символов искомой для слияния
                    if i + 1 < len(word) and (word[i], word[i + 1]) == merge:
                        # Объединяем пару в один токен
                        new_word.append(merge[0] + merge[1])
                        i += 2  # Пропускаем оба символа, поскольку они были объединены
                    else:
                        # Добавляем текущий символ в новое слово
                        new_word.append(word[i])
                        i += 1
                new_subwords.append(new_word)
            subwords = new_subwords

        # Преобразуем итоговые токены в строки и заменяем неизвестные токены на UNK
        result_tokens = []
        for word in subwords:
            result_tokens.extend([''.join(t) for t in word])

        # Замена неизвестных токенов на UNK
        return [t if self.vocab[t] else self.unk for t in result_tokens]

    def detokenize(self, subwords: List[str]) -> str:
        """
        Join tokens and remove UNK tokens.

        Args:
            subwords: List of tokens to join

        Returns:
            Reconstructed text string
        """
        return ''.join(token for token in subwords if token != self.unk)

    def encode(self, text: str) -> List[int]:
        """
        Convert text to token ids.

        Args:
            text: Input text string

        Returns:
            List of token ids
        """
        if self._token_to_id is None:
            # Lazily initialize token-to-id mapping
            self._token_to_id = {token: i for i, token in enumerate(self.vocab)}

        tokens = self.tokenize(text)
        return [self._token_to_id.get(token, self._token_to_id[self.unk]) for token in tokens]

    def decode(self, token_ids: List[int]) -> str:
        """
        Convert token ids back to text.

        Args:
            token_ids: List of token ids

        Returns:
            Reconstructed text string
        """
        if self._id_to_token is None:
            # Lazily initialize id-to-token mapping
            self._id_to_token = {i: token for i, token in enumerate(self.vocab)}

        tokens = [self._id_to_token.get(id, self.unk) for id in token_ids]
        return self.detokenize(tokens)

In [63]:
import pytest

from collections import Counter


def simple_tokenizer():
    return BPETokenizer(vocab_size=10)

def fitted_tokenizer():
    tokenizer = BPETokenizer(vocab_size=150)
    corpus = ["hello world", "hello there", "world peace"]
    tokenizer.fit(corpus)
    return tokenizer

def test_init():
    tokenizer = BPETokenizer(vocab_size=100)
    assert tokenizer.vocab_size == 100
    assert tokenizer.unk == DEFAULT_UNK
    assert tokenizer.pad == DEFAULT_PAD

def test_init_invalid_vocab_size():
    with pytest.raises(ValueError):
        BPETokenizer(vocab_size=0)

def test_fit(simple_tokenizer):
    corpus = ["hello"]
    vocab = simple_tokenizer.fit(corpus)
    assert isinstance(vocab, Counter)
    assert DEFAULT_UNK in vocab
    assert DEFAULT_PAD in vocab
    assert 'h' in vocab
    assert 'e' in vocab

def test_tokenize_without_fit(simple_tokenizer):
    with pytest.raises(ValueError):
        simple_tokenizer.tokenize("hello")

def test_tokenize(fitted_tokenizer):
    tokens = fitted_tokenizer.tokenize("hello")
    assert isinstance(tokens, list)
    assert all(isinstance(token, str) for token in tokens)
    assert all(token in fitted_tokenizer.vocab or token == DEFAULT_UNK for token in tokens)

def test_detokenize(fitted_tokenizer):
    tokens = fitted_tokenizer.tokenize("hello")
    text = fitted_tokenizer.detokenize(tokens)
    assert isinstance(text, str)
    assert text == "hello"

def test_encode_decode(fitted_tokenizer):
    original = "hello world"
    encoded = fitted_tokenizer.encode(original)
    assert isinstance(encoded, list)
    assert all(isinstance(id, int) for id in encoded)

    decoded = fitted_tokenizer.decode(encoded)
    assert decoded == original

def test_unknown_tokens(fitted_tokenizer):
    tokens = fitted_tokenizer.tokenize("xyz123")  # Characters not in training data
    assert DEFAULT_UNK in tokens

def test_empty_input(fitted_tokenizer):
    assert fitted_tokenizer.tokenize("") == []
    assert fitted_tokenizer.detokenize([]) == ""
    assert fitted_tokenizer.encode("") == []
    assert fitted_tokenizer.decode([]) == ""

def test_lorem_ipsum_tokenize():
    text = "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua."
    tokenizer = BPETokenizer(vocab_size=1000)
    corpus = [ text ] * 1000
    tokenizer.fit(corpus)
    tokens = tokenizer.tokenize("Lorem ipsum dolor sit amet")
    assert isinstance(tokens, list)
    assert all(isinstance(token, str) for token in tokens)
    assert all(token in tokenizer.vocab or token == DEFAULT_UNK for token in tokens)

def test_fitted_tokenizer_vocab(fitted_tokenizer):

    assert "hello " in fitted_tokenizer.vocab
    assert "world" in fitted_tokenizer.vocab

In [64]:
test_init()

In [65]:
test_init_invalid_vocab_size()

In [66]:
s = simple_tokenizer()
test_fit(s)

In [67]:
test_tokenize_without_fit(s)

In [68]:
f = fitted_tokenizer()

In [69]:
test_tokenize(f)

In [70]:
test_detokenize(f)

In [71]:
test_encode_decode(f)

In [72]:
test_unknown_tokens(f)

In [73]:
test_empty_input(f)

In [77]:
test_lorem_ipsum_tokenize()

In [75]:
test_fitted_tokenizer_vocab(f)

In [76]:
text = "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua."
len(text)

123